U-NET

Loading the drive

In [1]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')

Mounted at /content/drive


U Net Training




In [5]:
import torch.optim as optim

# 1. Definición del Dispositivo (GPU si está disponible)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 2. Función de Pérdida Especializada (Dice Loss)
class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice = (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)
        return 1 - dice

criterion = DiceLoss()

# 3. Optimizador Adam
# lr (learning rate): qué tan rápido aprende. 0.0001 es un valor estable para U-Net.
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 4. Guardado de Pesos
# Ruta donde se almacenará el cerebro de la IA conforme progrese
weights_path = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E5/Models/Weights/best_model.pth"

print(f"Configuración lista. Entrenando en: {device}")

Configuración lista. Entrenando en: cpu


Training Loop

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import os

# 1. Clase DiceLoss Corregida (Ajuste de Continuidad)
class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        # .contiguous() asegura que los parches extraídos sean compatibles con .view()
        inputs = inputs.contiguous().view(-1)
        targets = targets.contiguous().view(-1)

        intersection = (inputs * targets).sum()
        dice = (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return 1 - dice

# 2. Configuración del Entorno y Pesos
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights_path = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E5/Models/Weights/best_model.pth"

# 3. Hiperparámetros de Entrenamiento
EPOCHS = 5
PATCH_SIZE = 512 # Tamaño del fragmento para no saturar la RAM/GPU
best_loss = float('inf')

# 4. Loop de Entrenamiento
print(f"Iniciando entrenamiento en {device}...")
model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0
    start_time = time.time()

    for batch_idx, (data, target) in enumerate(loader):
        # Extracción de Parche Aleatorio (Cruce de Datos)
        h, w = data.shape[2], data.shape[3]
        top = torch.randint(0, h - PATCH_SIZE, (1,))
        left = torch.randint(0, w - PATCH_SIZE, (1,))

        data_patch = data[:, :, top:top+PATCH_SIZE, left:left+PATCH_SIZE].to(device)
        target_patch = target[:, :, top:top+PATCH_SIZE, left:left+PATCH_SIZE].to(device)

        # Ciclo de Optimización
        optimizer.zero_grad()
        output = model(data_patch)
        loss = criterion(output, target_patch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Cálculo de métricas y guardado
    avg_loss = epoch_loss / len(loader)
    elapsed = time.time() - start_time
    print(f"Época {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Tiempo: {elapsed:.2f}s")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), weights_path)
        print(f"--> [NUEVO RÉCORD] Modelo guardado en Drive.")

print("\nValidación estructural completa. La red ha asimilado la topografía de Veracruz.")

Iniciando entrenamiento en cpu...
Época 1/5 | Loss: 0.7118 | Tiempo: 84.14s
--> [NUEVO RÉCORD] Modelo guardado en Drive.
Época 2/5 | Loss: 0.6050 | Tiempo: 76.30s
--> [NUEVO RÉCORD] Modelo guardado en Drive.
Época 3/5 | Loss: 0.5658 | Tiempo: 76.56s
--> [NUEVO RÉCORD] Modelo guardado en Drive.
Época 4/5 | Loss: 0.5549 | Tiempo: 76.29s
--> [NUEVO RÉCORD] Modelo guardado en Drive.
Época 5/5 | Loss: 0.4756 | Tiempo: 76.38s
--> [NUEVO RÉCORD] Modelo guardado en Drive.

Validación estructural completa. La red ha asimilado la topografía de Veracruz.
